# LSTM Panel — Prediksi ASI Eksklusif

**Pipeline lengkap:** Data Preprocessing → EDA → Feature Engineering → Model Building → Training → Evaluasi → SHAP Explainability

Menggunakan dataset `data_master_2021_2024_scaled.csv` (24 Puskesmas, 2021–2024, 1152 baris).

---
### Colab Setup

Jalankan sel berikut untuk:
1. Install dependencies
2. Mount Google Drive
3. Upload dataset

**Cara upload dataset:** Jalankan sel upload → pilih file `data_master_2021_2024_scaled.csv` dari lokal.

In [ ]:
# Install dependencies (Colab)
!pip install -q shap seaborn matplotlib pandas numpy tensorflow scikit-learn joblib
print('Dependencies installed.')

In [ ]:
# Mount Google Drive (opsional — untuk akses file dari Drive)
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

In [ ]:
# Upload dataset dari lokal
from google.colab import files
uploaded = files.upload()

# Cari file CSV yang diupload
import os
csv_files = [f for f in uploaded.keys() if f.endswith('.csv')]
if csv_files:
    CSV_PATH = csv_files[0]
    print(f'Dataset: {CSV_PATH} ({os.path.getsize(CSV_PATH)} bytes)')
else:
    # Fallback ke path Drive
    CSV_PATH = '/content/drive/MyDrive/data_master_2021_2024_scaled.csv'
    print(f'Menggunakan path Drive: {CSV_PATH}')

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import joblib

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, confusion_matrix

import shap

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
np.random.seed(42)
tf.random.set_seed(42)

# Cek path file
if not os.path.exists(CSV_PATH):
    # Coba path alternatif
    alt = '/content/drive/MyDrive/ColabNotebooks/data_master_2021_2024_scaled.csv'
    if os.path.exists(alt):
        CSV_PATH = alt
    else:
        # Upload ulang jika file tidak ditemukan
        from google.colab import files
        uploaded = files.upload()
        csv_files = [f for f in uploaded.keys() if f.endswith('.csv')]
        if csv_files:
            CSV_PATH = csv_files[0]
        else:
            raise FileNotFoundError('Dataset tidak ditemukan. Upload file CSV.')

print(f'CSV_PATH = {CSV_PATH}')
print('Semua library berhasil diimport.')

---
## 1. Data Loading & Inspection

In [ ]:
df = pd.read_csv(CSV_PATH)
df['Tanggal'] = pd.to_datetime(df['Tanggal'])
df = df.sort_values(['Puskesmas', 'Tanggal']).reset_index(drop=True)

print(f'Dataset: {len(df)} baris, {df.columns.tolist()}')
print(f'Periode: {df["Tanggal"].min().date()} s.d. {df["Tanggal"].max().date()}')
print(f'Puskesmas: {df["Puskesmas"].nunique()} buah')
print(f'Kecamatan: {df["Kecamatan"].nunique()} buah')
print(f'\nTahun: {sorted(df["Tanggal"].dt.year.unique())}')
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

---
## 2. Exploratory Data Analysis (EDA)

### 2.1 Target Distribution

In [ ]:
def get_segment(val):
    if val >= 80: return 'Tinggi'
    if val >= 60: return 'Sedang'
    return 'Rendah'

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(df['Persentase_Cakupan'], bins=30, color='#2563eb', edgecolor='white', alpha=0.8)
axes[0].axvline(80, color='#10b981', ls='--', lw=2, label='Threshold Tinggi (80)')
axes[0].axvline(60, color='#f59e0b', ls='--', lw=2, label='Threshold Sedang (60)')
axes[0].set_xlabel('Persentase Cakupan (%)')
axes[0].set_ylabel('Frekuensi')
axes[0].set_title('Distribusi Target')
axes[0].legend()

df['Tahun'] = df['Tanggal'].dt.year
sns.boxplot(data=df, x='Tahun', y='Persentase_Cakupan', palette='Blues', ax=axes[1])
axes[1].axhline(80, color='#10b981', ls='--', lw=1.5)
axes[1].set_title('Distribusi per Tahun')

seg_counts = df['Persentase_Cakupan'].apply(get_segment).value_counts()
colors_seg = {'Tinggi': '#10b981', 'Sedang': '#f59e0b', 'Rendah': '#ef4444'}
axes[2].pie(
    seg_counts.values, labels=[f'{k}\n({v})' for k, v in seg_counts.items()],
    colors=[colors_seg[k] for k in seg_counts.index],
    autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10}
)
axes[2].set_title('Segmen Cakupan')

plt.tight_layout()
plt.show()
print(seg_counts.to_string())

### 2.2 Correlation Matrix

In [ ]:
feat_cols = ['Jumlah_ASI_Eksklusif', 'Jumlah_Bayi_6_Bulan', 'Persentase_Cakupan']
corr = df[feat_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Korelasi Antar Variabel')
plt.tight_layout()
plt.show()

print('\nKorelasi dengan target (Persentase_Cakupan):')
for c in feat_cols:
    if c != 'Persentase_Cakupan':
        r = df[c].corr(df['Persentase_Cakupan'])
        print(f'  {c:30s}  r = {r:+.4f}')

### 2.3 Time Series per Puskesmas

In [ ]:
pkm_list = sorted(df['Puskesmas'].unique())
n_pkm = len(pkm_list)
n_cols = 6
n_rows = int(np.ceil(n_pkm / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 2.8 * n_rows), sharey=True)
axes = axes.flatten()

for i, pkm in enumerate(pkm_list):
    sub = df[df['Puskesmas'] == pkm].sort_values('Tanggal')
    axes[i].plot(sub['Tanggal'], sub['Persentase_Cakupan'], color='#2563eb', lw=1.2)
    axes[i].axhline(80, color='#10b981', ls='--', lw=0.8, alpha=0.5)
    axes[i].axhline(60, color='#f59e0b', ls='--', lw=0.8, alpha=0.5)
    axes[i].set_title(pkm, fontsize=8)
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Time Series Persentase Cakupan per Puskesmas (2021–2024)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### 2.4 Rata-rata per Puskesmas

In [ ]:
pkm_mean = df.groupby('Puskesmas')['Persentase_Cakupan'].agg(['mean', 'std', 'min', 'max']).sort_values('mean')

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.barh(range(len(pkm_mean)), pkm_mean['mean'], xerr=pkm_mean['std'],
               color='#2563eb', alpha=0.8, ecolor='gray', capsize=3)
ax.axvline(80, color='#10b981', ls='--', lw=2, label='Tinggi (≥80)')
ax.axvline(60, color='#f59e0b', ls='--', lw=2, label='Sedang (≥60)')
ax.set_yticks(range(len(pkm_mean)))
ax.set_yticklabels(pkm_mean.index, fontsize=7)
ax.set_xlabel('Persentase Cakupan (%)')
ax.set_title('Rata-rata + Std Persentase Cakupan per Puskesmas')
ax.legend()
plt.tight_layout()
plt.show()
pkm_mean

### 2.5 Analisis per Kecamatan

In [ ]:
kec_stats = df.groupby('Kecamatan').agg(
    n_puskesmas=('Puskesmas', 'nunique'),
    mean_cakupan=('Persentase_Cakupan', 'mean'),
    std_cakupan=('Persentase_Cakupan', 'std')
).sort_values('mean_cakupan')

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = ['#10b981' if m >= 80 else '#f59e0b' for m in kec_stats['mean_cakupan']]
ax.barh(kec_stats.index, kec_stats['mean_cakupan'], xerr=kec_stats['std_cakupan'],
        color=colors, alpha=0.8, ecolor='gray', capsize=3)
ax.axvline(80, color='#10b981', ls='--', lw=2)
ax.set_xlabel('Rata-rata Persentase Cakupan (%)')
ax.set_title('Rata-rata Cakupan per Kecamatan')
plt.tight_layout()
plt.show()
kec_stats

---
## 3. Feature Engineering

Fitur yang dibuat:
- **Rasio_ASI_Bayi** = Jumlah_ASI_Eksklusif / Jumlah_Bayi_6_Bulan (r=0.89 dengan target)
- **Lag1/2/3_Target** — autoregressive features (t-1, t-2, t-3)
- **Month_Sin, Month_Cos** — seasonal encoding
- **Year_Trend** — normalized year (0→1)

In [ ]:
WINDOW_SIZE = 12
N_FEATURES = 8
FEATURE_NAMES = [
    'Jumlah_ASI_Eksklusif',
    'Rasio_ASI_Bayi',
    'Lag1_Target',
    'Lag2_Target',
    'Lag3_Target',
    'Month_Sin',
    'Month_Cos',
    'Year_Trend',
]

def engineer_features(df_raw):
    result = df_raw.copy()
    result = result.sort_values(['Puskesmas', 'Tanggal']).reset_index(drop=True)
    result['Rasio_ASI_Bayi'] = result['Jumlah_ASI_Eksklusif'] / (result['Jumlah_Bayi_6_Bulan'] + 1e-8)
    for lag in [1, 2, 3]:
        result[f'Lag{lag}_Target'] = result.groupby('Puskesmas')['Persentase_Cakupan'].shift(lag)
    result['Month'] = result['Tanggal'].dt.month
    result['Month_Sin'] = np.sin(2 * np.pi * result['Month'] / 12)
    result['Month_Cos'] = np.cos(2 * np.pi * result['Month'] / 12)
    result['Year'] = result['Tanggal'].dt.year
    result['Year_Trend'] = (result['Year'] - 2021) / 3.0
    result = result.dropna().reset_index(drop=True)
    return result

df_feat = engineer_features(df)
print(f'After feature engineering: {len(df_feat)} baris')
print(f'Fitur ({N_FEATURES}): {FEATURE_NAMES}')
df_feat.head()

In [ ]:
corr_with_target = {}
for f in FEATURE_NAMES:
    corr_with_target[f] = df_feat[f].corr(df_feat['Persentase_Cakupan'])

corr_series = pd.Series(corr_with_target).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors_corr = ['#10b981' if v >= 0 else '#ef4444' for v in corr_series.values]
ax.barh(range(len(corr_series)), corr_series.values, color=colors_corr, alpha=0.8)
ax.set_yticks(range(len(corr_series)))
ax.set_yticklabels(corr_series.index)
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('Korelasi (r) dengan Persentase_Cakupan')
ax.set_title('Feature Correlation with Target')
for i, (k, v) in enumerate(corr_series.items()):
    ax.text(v + (0.02 if v >= 0 else -0.12), i, f'{v:+.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

r_val = df_feat['Rasio_ASI_Bayi'].corr(df_feat['Persentase_Cakupan'])
axes[0].scatter(df_feat['Rasio_ASI_Bayi'], df_feat['Persentase_Cakupan'], alpha=0.3, s=15, c='#2563eb')
z = np.polyfit(df_feat['Rasio_ASI_Bayi'], df_feat['Persentase_Cakupan'], 1)
p = np.poly1d(z)
x_line = np.linspace(df_feat['Rasio_ASI_Bayi'].min(), df_feat['Rasio_ASI_Bayi'].max(), 100)
axes[0].plot(x_line, p(x_line), 'r--', lw=2)
axes[0].set_xlabel('Rasio_ASI_Bayi')
axes[0].set_ylabel('Persentase_Cakupan (%)')
axes[0].set_title(f'Rasio_ASI_Bayi vs Target (r={r_val:+.3f})')

for idx, pkm in enumerate(sorted(df_feat['Puskesmas'].unique())):
    sub = df_feat[df_feat['Puskesmas'] == pkm]
    axes[1].scatter(sub['Rasio_ASI_Bayi'], sub['Persentase_Cakupan'], alpha=0.5, s=10, label=pkm if idx < 5 else '')
axes[1].set_xlabel('Rasio_ASI_Bayi')
axes[1].set_ylabel('Persentase_Cakupan (%)')
axes[1].set_title('Per-Puskesmas: Rasio vs Target')
axes[1].legend(loc='best', fontsize=6, ncol=2)

plt.tight_layout()
plt.show()

slopes = []
for pkm in sorted(df_feat['Puskesmas'].unique()):
    sub = df_feat[df_feat['Puskesmas'] == pkm]
    if len(sub) > 5:
        zz = np.polyfit(sub['Rasio_ASI_Bayi'], sub['Persentase_Cakupan'], 1)
        slopes.append({'Puskesmas': pkm, 'slope': zz[0], 'intercept': zz[1]})
slopes_df = pd.DataFrame(slopes)
print(f'Slope Rasio->Target per puskesmas: min={slopes_df["slope"].min():.1f}, max={slopes_df["slope"].max():.1f}, mean={slopes_df["slope"].mean():.1f}')

---
## 4. Create Sequences & Split

In [ ]:
all_X, all_y, all_dates = [], [], []
for pkm in sorted(df_feat['Puskesmas'].unique()):
    pkm_data = df_feat[df_feat['Puskesmas'] == pkm].sort_values('Tanggal')
    dates = pkm_data['Tanggal'].values
    X_raw = pkm_data[FEATURE_NAMES].values.astype(np.float32)
    y_raw = pkm_data['Persentase_Cakupan'].values.astype(np.float32)
    if len(X_raw) < WINDOW_SIZE + 1:
        continue
    for i in range(len(X_raw) - WINDOW_SIZE):
        all_X.append(X_raw[i:i + WINDOW_SIZE])
        all_y.append(y_raw[i + WINDOW_SIZE - 1])
        all_dates.append(dates[i + WINDOW_SIZE - 1])

X_all = np.array(all_X, dtype=np.float32)
y_all = np.array(all_y, dtype=np.float32)
dates_all = np.array(all_dates)
print(f'Total sequences: {X_all.shape}, Target range: {y_all.min():.1f}% - {y_all.max():.1f}%')

In [ ]:
scaler_X = StandardScaler()
s = X_all.shape
X_scaled = scaler_X.fit_transform(X_all.reshape(-1, s[-1])).reshape(s)
scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y_all.reshape(-1, 1)).flatten()
print('Scaler_X mean:', scaler_X.mean_)
print('Scaler_y mean:', scaler_y.mean_[0], ', scale:', scaler_y.scale_[0])

In [ ]:
sort_idx = np.argsort(dates_all)
X_sorted = X_scaled[sort_idx]
y_sorted = y_scaled[sort_idx]
dates_sorted = dates_all[sort_idx]

n_total = len(X_sorted)
split_idx = int(n_total * 0.8)
split_date = dates_sorted[split_idx]

X_tr = X_sorted[:split_idx]
y_tr = y_sorted[:split_idx]
X_v = X_sorted[split_idx:]
y_v = y_sorted[split_idx:]

print(f'Train: {X_tr.shape}')
print(f'Val:   {X_v.shape}')
print(f'Split date: {split_date}')
print(f'Train target: {scaler_y.inverse_transform(y_tr.reshape(-1,1)).min():.1f} - {scaler_y.inverse_transform(y_tr.reshape(-1,1)).max():.1f}')
print(f'Val target:   {scaler_y.inverse_transform(y_v.reshape(-1,1)).min():.1f} - {scaler_y.inverse_transform(y_v.reshape(-1,1)).max():.1f}')

---
## 5. Model Building

**Non-temporal:** hanya mengambil timestep terakhir → Dense layers.
Rasio_ASI_Bayi (r=0.89) dominan dan hubungannya instan, jadi LSTM tidak diperlukan.

In [ ]:
def build_model(input_shape):
    inputs = Input(shape=input_shape, name='input')
    x = Dense(24, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4), name='dense_1')(inputs[:, -1, :])
    x = BatchNormalization(name='bn_1')(x)
    x = Dropout(0.15, name='dropout_1')(x)
    x = Dense(12, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4), name='dense_2')(x)
    outputs = Dense(1, name='output')(x)
    model = Model(inputs=inputs, outputs=outputs, name='lstm_panel_v6')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='huber', metrics=['mae'])
    return model

model = build_model((WINDOW_SIZE, N_FEATURES))
model.summary()

---
## 6. Training

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=50, min_delta=1e-4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=15, min_lr=1e-5, verbose=1),
]

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_v, y_v),
    epochs=300, batch_size=32,
    callbacks=callbacks, verbose=1,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(history.history['loss'], label='Train Loss', color='#2563eb')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#10b981')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (Huber)')
axes[0].set_title('Training vs Validation Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train MAE', color='#2563eb')
axes[1].plot(history.history['val_mae'], label='Val MAE', color='#10b981')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE')
axes[1].set_title('Training vs Validation MAE'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_epoch = np.argmin(history.history['val_loss']) + 1
print(f'Best epoch: {best_epoch}, Best val_loss: {min(history.history["val_loss"]):.6f}')

---
## 7. Evaluasi

### 7.1 Overall Metrics

In [ ]:
y_tr_pred = scaler_y.inverse_transform(model.predict(X_tr, verbose=0).reshape(-1, 1)).flatten()
y_tr_actual = scaler_y.inverse_transform(y_tr.reshape(-1, 1)).flatten()
y_v_pred = scaler_y.inverse_transform(model.predict(X_v, verbose=0).reshape(-1, 1)).flatten()
y_v_actual = scaler_y.inverse_transform(y_v.reshape(-1, 1)).flatten()

train_r2 = r2_score(y_tr_actual, y_tr_pred)
train_mae = mean_absolute_error(y_tr_actual, y_tr_pred)
val_r2 = r2_score(y_v_actual, y_v_pred)
val_mae = mean_absolute_error(y_v_actual, y_v_pred)

y_v_seg_actual = np.array([get_segment(v) for v in y_v_actual])
y_v_seg_pred = np.array([get_segment(v) for v in y_v_pred])
seg_acc = accuracy_score(y_v_seg_actual, y_v_seg_pred)

print('=' * 50)
print('EVALUATION RESULTS')
print('=' * 50)
print(f'Train R²:  {train_r2:.4f}')
print(f'Train MAE: {train_mae:.2f}%')
print(f'Val R²:    {val_r2:.4f}')
print(f'Val MAE:   {val_mae:.2f}%')
print(f'Segment Accuracy (Val): {seg_acc:.2%}')
print(f'Prediction Std: {np.std(y_v_pred):.2f}% (actual: {np.std(y_v_actual):.2f}%)')

### 7.2 Predicted vs Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

axes[0].scatter(y_tr_actual, y_tr_pred, alpha=0.4, s=15, c='#2563eb')
axes[0].plot([70, 90], [70, 90], 'r--', lw=1.5, label='Ideal')
axes[0].set_xlabel('Actual (%)'); axes[0].set_ylabel('Predicted (%)')
axes[0].set_title(f'Train (R²={train_r2:.4f}, MAE={train_mae:.2f}%)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_v_actual, y_v_pred, alpha=0.4, s=15, c='#10b981')
axes[1].plot([70, 90], [70, 90], 'r--', lw=1.5, label='Ideal')
axes[1].set_xlabel('Actual (%)'); axes[1].set_ylabel('Predicted (%)')
axes[1].set_title(f'Validation (R²={val_r2:.4f}, MAE={val_mae:.2f}%)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7.3 Residual Analysis

In [ ]:
residuals = y_v_actual - y_v_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(residuals, bins=25, color='#10b981', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', ls='--', lw=1.5)
axes[0].set_xlabel('Residual (Actual - Predicted, %)'); axes[0].set_ylabel('Frekuensi')
axes[0].set_title(f'Residual Distribution (std={np.std(residuals):.2f}%)')

axes[1].scatter(y_v_pred, residuals, alpha=0.4, s=15, c='#2563eb')
axes[1].axhline(0, color='red', ls='--', lw=1.5)
axes[1].set_xlabel('Predicted (%)'); axes[1].set_ylabel('Residual (%)')
axes[1].set_title('Residual vs Predicted'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Residual: mean={np.mean(residuals):+.4f}%, std={np.std(residuals):.4f}%, |res|<1%: {(np.abs(residuals) < 1).mean():.2%}')

### 7.4 Segment Confusion Matrix

In [ ]:
cm = confusion_matrix(y_v_seg_actual, y_v_seg_pred, labels=['Tinggi', 'Sedang', 'Rendah'])

fig, ax = plt.subplots(figsize=(5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tinggi', 'Sedang', 'Rendah'],
            yticklabels=['Tinggi', 'Sedang', 'Rendah'],
            ax=ax, cbar_kws={'shrink': 0.8})
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Segment Confusion Matrix (Acc={seg_acc:.2%})')
plt.tight_layout()
plt.show()

### 7.5 Per-Puskesmas Evaluation

In [ ]:
pkm_results = []
for pkm in sorted(df_feat['Puskesmas'].unique()):
    pkm_data = df_feat[df_feat['Puskesmas'] == pkm].sort_values('Tanggal')
    X_raw = pkm_data[FEATURE_NAMES].values.astype(np.float32)
    y_raw = pkm_data['Persentase_Cakupan'].values.astype(np.float32)
    if len(X_raw) < WINDOW_SIZE + 1:
        continue
    X_p, y_p = [], []
    for i in range(len(X_raw) - WINDOW_SIZE):
        X_p.append(X_raw[i:i + WINDOW_SIZE])
        y_p.append(y_raw[i + WINDOW_SIZE - 1])
    X_p = np.array(X_p, dtype=np.float32)
    y_p = np.array(y_p, dtype=np.float32)
    s_p = X_p.shape
    X_p_scaled = scaler_X.transform(X_p.reshape(-1, s_p[-1])).reshape(s_p)
    y_pred_p = scaler_y.inverse_transform(model.predict(X_p_scaled, verbose=0).reshape(-1, 1)).flatten()

    pkm_results.append({
        'Puskesmas': pkm, 'Kecamatan': pkm_data['Kecamatan'].iloc[0], 'n': len(y_p),
        'R²': round(r2_score(y_p, y_pred_p), 4),
        'MAE': round(mean_absolute_error(y_p, y_pred_p), 2),
        'Seg': round(accuracy_score(np.array([get_segment(v) for v in y_p]), np.array([get_segment(v) for v in y_pred_p])), 2),
    })

df_res = pd.DataFrame(pkm_results)

fig, ax = plt.subplots(figsize=(12, 5.5))
colors_bar = ['#10b981' if r >= 0.80 else '#f59e0b' if r >= 0.60 else '#ef4444' for r in df_res['R²']]
ax.barh(range(len(df_res)), df_res['R²'], color=colors_bar, alpha=0.85)
ax.axvline(0.80, color='#10b981', ls='--', lw=2, label='Target R²=0.80')
ax.axvline(0.60, color='#f59e0b', ls='--', lw=1, alpha=0.5)
ax.set_yticks(range(len(df_res)))
ax.set_yticklabels(df_res['Puskesmas'], fontsize=7)
ax.set_xlabel('R² Score'); ax.set_title('Per-Puskesmas R² Score'); ax.legend()
for i, (_, row) in enumerate(df_res.iterrows()):
    ax.text(row['R²'] + 0.01, i, f'{row["R²"]:.3f} ({row["Seg"]:.0%})', va='center', fontsize=7)
plt.tight_layout()
plt.show()

print(f'Mean R²: {df_res["R²"].mean():.4f}, Mean MAE: {df_res["MAE"].mean():.2f}%')
print(f'R²>=0.80: {(df_res["R²"]>=0.80).sum()}/{len(df_res)}, R²>=0.60: {(df_res["R²"]>=0.60).sum()}/{len(df_res)}')

### 7.6 Time Series Prediction (Sample)

In [ ]:
sample_pkms = sorted(df_feat['Puskesmas'].unique())[:4]
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
axes = axes.flatten()

for ax_idx, pkm in enumerate(sample_pkms):
    sub = df_feat[df_feat['Puskesmas'] == pkm].sort_values('Tanggal').reset_index(drop=True)
    X_raw = sub[FEATURE_NAMES].values.astype(np.float32)
    y_raw = sub['Persentase_Cakupan'].values.astype(np.float32)
    if len(X_raw) < WINDOW_SIZE + 1: continue
    X_p, y_p, dates_p = [], [], []
    for i in range(len(X_raw) - WINDOW_SIZE):
        X_p.append(X_raw[i:i + WINDOW_SIZE])
        y_p.append(y_raw[i + WINDOW_SIZE - 1])
        dates_p.append(sub['Tanggal'].iloc[i + WINDOW_SIZE - 1])
    X_p = np.array(X_p, dtype=np.float32)
    s_p = X_p.shape
    X_p_scaled = scaler_X.transform(X_p.reshape(-1, s_p[-1])).reshape(s_p)
    y_pred_p = scaler_y.inverse_transform(model.predict(X_p_scaled, verbose=0).reshape(-1, 1)).flatten()

    axes[ax_idx].plot(dates_p, np.array(y_p), 'o-', label='Actual', color='#2563eb', lw=1.5, ms=4)
    axes[ax_idx].plot(dates_p, y_pred_p, 's--', label='Predicted', color='#10b981', lw=1.5, ms=4)
    axes[ax_idx].axhline(80, color='gray', ls=':', lw=0.8, alpha=0.5)
    axes[ax_idx].set_title(pkm); axes[ax_idx].legend(fontsize=8)

plt.suptitle('Actual vs Predicted — Sample Puskesmas', fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. SHAP Explainability

Menggunakan **DeepExplainer** dengan **wrapper model** (8 fitur dari timestep terakhir).

In [ ]:
inp_wrapper = tf.keras.layers.Input(shape=(N_FEATURES,), name='shap_input')
x_w = model.get_layer('dense_1')(inp_wrapper)
x_w = model.get_layer('bn_1')(x_w)
x_w = model.get_layer('dropout_1')(x_w, training=False)
x_w = model.get_layer('dense_2')(x_w)
out_w = model.get_layer('output')(x_w)
wrapper_model = tf.keras.Model(inputs=inp_wrapper, outputs=out_w, name='shap_wrapper')
wrapper_model.compile()
print('Wrapper input shape:', wrapper_model.input_shape)

In [ ]:
bg_idx = np.random.choice(len(X_tr), min(100, len(X_tr)), replace=False)
bg_data = X_tr[bg_idx, -1, :]
explainer = shap.DeepExplainer(wrapper_model, bg_data)

expected_value_scaled = float(np.mean(wrapper_model.predict(bg_data, verbose=0)))
expected_value_pct = scaler_y.inverse_transform([[expected_value_scaled]])[0, 0]
print(f'Expected value (scaled): {expected_value_scaled:.4f}')
print(f'Expected value (%): {expected_value_pct:.2f}%')

In [ ]:
n_shap_samples = 5
X_shap = X_v[:n_shap_samples, -1, :]
shap_values = explainer.shap_values(X_shap)
if isinstance(shap_values, list): shap_arr = shap_values[0]
else: shap_arr = shap_values
print(f'SHAP values shape: {shap_arr.shape}')

for i in range(N_FEATURES):
    val_pct = shap_arr[0, i] * scaler_y.scale_[0]
    print(f'  {FEATURE_NAMES[i]:25s}  {shap_arr[0,i]:+.4f} (scaled) = {val_pct:+.4f}%')

In [ ]:
# Verify: ev + sum(SHAP) == prediction
for sidx in range(min(3, n_shap_samples)):
    sum_shap = float(np.sum(shap_arr[sidx]))
    pred_scaled = float(model.predict(X_v[sidx:sidx+1], verbose=0)[0, 0])
    ev_plus_sum = expected_value_scaled + sum_shap
    ok = '✓' if abs(ev_plus_sum - pred_scaled) < 0.01 else '✗'
    print(f'Sample {sidx}: ev={expected_value_scaled:.4f} + sum={sum_shap:+.4f} = {ev_plus_sum:.4f} vs pred={pred_scaled:.4f} {ok}')

In [ ]:
mean_abs_shap = np.mean(np.abs(shap_arr), axis=0) * scaler_y.scale_[0]
mean_shap = np.mean(shap_arr, axis=0) * scaler_y.scale_[0]
imp_df = pd.DataFrame({'Feature': FEATURE_NAMES, 'Mean |SHAP|': mean_abs_shap, 'Mean SHAP': mean_shap}).sort_values('Mean |SHAP|', ascending=True)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(imp_df['Feature'], imp_df['Mean |SHAP|'], color='#2563eb', alpha=0.8)
ax.set_xlabel('Mean |SHAP| (% point)'); ax.set_title('SHAP Feature Importance')
for i, (_, row) in enumerate(imp_df.iterrows()):
    ax.text(row['Mean |SHAP|'] + 0.05, i, f'{row["Mean |SHAP|"]:.2f}% ({row["Mean SHAP"]:+.2f}%)', va='center', fontsize=8)
plt.tight_layout()
plt.show()
imp_df

In [ ]:
shap.summary_plot(shap_arr, X_shap, feature_names=FEATURE_NAMES, show=True)

In [ ]:
shap.waterfall_plot(
    shap.Explanation(
        values=shap_arr[0], base_values=expected_value_scaled,
        data=X_shap[0], feature_names=FEATURE_NAMES
    ), show=True
)

---
## 9. Save Model & Artifacts

Hasil akan disimpan ke Google Drive (jika sudah di-mount) atau ke folder lokal Colab.

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/models_asi' if os.path.exists('/content/drive/MyDrive') else 'saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save(f'{SAVE_DIR}/model_lstm_panel.keras')
joblib.dump(scaler_X, f'{SAVE_DIR}/scaler_X.pkl')
joblib.dump(scaler_y, f'{SAVE_DIR}/scaler_Y.pkl')
np.save(f'{SAVE_DIR}/background_data.npy', X_tr[np.random.choice(len(X_tr), min(200, len(X_tr)), replace=False)])

print(f'Model & artifacts saved to {SAVE_DIR}/')

---
## 10. Summary

In [ ]:
print('=' * 60)
print('PIPELINE SUMMARY')
print('=' * 60)
print(f'Dataset: {len(df)} baris, {df["Puskesmas"].nunique()} puskesmas, {df["Kecamatan"].nunique()} kecamatan')
print(f'Features ({N_FEATURES}): {FEATURE_NAMES}')
print(f'Window: {WINDOW_SIZE} bulan')
print(f'\nTrain R²: {train_r2:.4f}, MAE: {train_mae:.2f}%')
print(f'Val   R²: {val_r2:.4f}, MAE: {val_mae:.2f}%')
print(f'Segment Acc: {seg_acc:.2%}')
print(f'\nDominant: Rasio_ASI_Bayi r={df_feat["Rasio_ASI_Bayi"].corr(df_feat["Persentase_Cakupan"]):.3f}')
print(f'\nSHAP:')
for _, row in imp_df.iterrows():
    print(f'  {row["Feature"]:25s} |SHAP|={row["Mean |SHAP|"]:.3f}% mean={row["Mean SHAP"]:+.3f}%')
print('\nDone. ✓')